# Craft My Book

Uses the `llm` package in `src/` (OpenAI provider) to draft book content.

In [1]:
import sys
from pathlib import Path


sys.path.insert(0, str(Path.cwd() / "src"))

from llm import get_client, chat
import config

In [2]:
client = get_client(config.LLM_PROVIDER)
model = config.LLM_MODEL

In [3]:
prompt = "Write an engaging opening paragraph for a book about ..."

response = chat(client, model, prompt)
print(response)

Certainly! What specific theme or subject would you like the opening paragraph to focus on?


## Module 1.2 — Speech Processing (Whisper)

Turns a lecture recording — video or audio — into a structured, timestamped,
domain-aware transcript (`src/ingestion/speech.py`, `src/ingestion/vocab.py`).
Requires the `ffmpeg` binary on PATH and `faster-whisper` installed
(`pip install -r requirements.txt`).

**Design**: `ingestion.base.Ingestor` is an abstract strategy — `ingest(source_path) ->
IngestedDocument` — that every source type implements (speech today; PDF/DOCX/PPTX/
image strategies for Pipeline A land the same way later). `SpeechIngestor` is the
speech strategy; `ingestion.registry.get_ingestor_class()` picks the right strategy
for a file by extension, so code that walks a folder of mixed sources never branches
on file type itself. `IngestedDocument`/`IngestedSegment` are the common output shape
every strategy produces, regardless of what ran underneath.

All the tunable choices — Whisper model size, LLM provider/model, output dir — live
in `src/config/config.yaml` (loaded once by `src/config/__init__.py`), not scattered
across cells/function signatures. Whisper currently defaults to `"small"` (fast,
fully local-cacheable) for local iteration; bump `whisper.model_size` to `"large-v3"`
in that YAML for real lecture-quality transcription once you have a stable connection
(and ideally a GPU) — see `ingestion/speech.py` for why domain accuracy matters there.

Flow: `extract_vocab` (LLM pass over slide titles/filenames/headings — vocab is
per-corpus, never hardcoded) → `extract_audio` (ffmpeg; handles video or audio
sources) → `transcribe_audio` (faster-whisper, vocab-primed, VAD-filtered, word
timestamps, no cross-segment conditioning) → `clean_transcript` (conservative LLM
pass — fixes terms/punctuation, changes nothing else) → `IngestedDocument` JSON.

In [4]:
from ingestion import extract_audio, extract_vocab, vocab_material_from_filenames, bootstrap_vocab_from_audio

# data/harvard-speech.wav: ~34s of real spoken English (public-domain Harvard sentences
# test corpus) - use this to smoke-test the pipeline locally before pointing it at a
# real lecture.
source_path = "data/harvard-speech.wav"

# Preferred: derive vocab from material that already exists around the recording
# (slide titles / filenames here; could also be PDF headings).
sibling_files: list[str] = []  # no slides for this sample -> falls through to bootstrap
vocab_material = vocab_material_from_filenames(sibling_files)
vocab = extract_vocab(vocab_material, client, model)

# Fallback: audio is the ONLY source (no slides/useful filenames) - bootstrap vocab from
# a fast draft transcription pass instead of skipping priming altogether.
if not vocab:
    audio_path = extract_audio(source_path)
    vocab = bootstrap_vocab_from_audio(audio_path, client, model)

vocab

/opt/homebrew/Caskroom/miniforge/base/envs/rl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['birch canoe',
 'depth',
 'chicken leg',
 'round bowls',
 'juice',
 'lemons',
 'punch',
 'box',
 'truck',
 'hogs',
 'chopped corn',
 'garbage',
 'hours',
 'stockings']

In [6]:
# Or, end to end via the strategy: registry picks the Ingestor for this file, ingest()
# runs source -> audio -> transcribe -> clean, save_document() writes the JSON. Once
# PDF/DOCX/image strategies exist, code that walks a mixed folder still looks like this.
from ingestion import get_ingestor_class, save_document

IngestorClass = get_ingestor_class(source_path)  # -> SpeechIngestor, by extension
ingestor = IngestorClass(client=client, clean_model=model, vocab=vocab)

document = ingestor.ingest(source_path)
out_path = save_document(document, config.TRANSCRIPTS_OUT_DIR)

print(f"{document.source_type}: {len(document.segments)} segments, {document.metadata['duration']:.0f}s -> {out_path}")
document.segments[0]

speech: 6 segments, 34s -> output/transcripts/harvard-speech.json


IngestedSegment(text='Birch canoe slid on the smooth planks. Glue the sheet to the dark blue background.', order=0, start=np.float64(0.11), end=np.float64(6.19), page=None, metadata={'words': [{'start': np.float64(0.11), 'end': np.float64(1.01), 'text': 'Birch'}, {'start': np.float64(1.01), 'end': np.float64(1.33), 'text': 'canoe'}, {'start': np.float64(1.33), 'end': np.float64(1.89), 'text': 'slid'}, {'start': np.float64(1.89), 'end': np.float64(2.07), 'text': 'on'}, {'start': np.float64(2.07), 'end': np.float64(2.23), 'text': 'the'}, {'start': np.float64(2.23), 'end': np.float64(2.51), 'text': 'smooth'}, {'start': np.float64(2.51), 'end': np.float64(3.05), 'text': 'planks.'}, {'start': np.float64(3.97), 'end': np.float64(4.49), 'text': 'Glue'}, {'start': np.float64(4.49), 'end': np.float64(4.71), 'text': 'the'}, {'start': np.float64(4.71), 'end': np.float64(4.97), 'text': 'sheet'}, {'start': np.float64(4.97), 'end': np.float64(5.13), 'text': 'to'}, {'start': np.float64(5.13), 'end': 